# Lab 2: Document Structure

Read the sentence pairs of the *SMTeuroparl* test set, compute their similarity using words and the Jaccard distance, and compare it with the gold standard using the Pearson correlation.

The words are obtained with two tokenizers, kept separate: **NLTK** (`word_tokenize`, the one in the lab slides) and **spaCy** (the tokenizer of `en_core_web_sm`). The next labs build on these two pipelines (Lab 3 adds lemmas, Lab 6 adds senses), so each library is followed from start to end.

## Setup

Import the libraries, and download the NLTK tokenizer and the spaCy model. The code for TextServer (FreeLing) is also here, but it is off (`RUN_TEXTSERVER` in the setup): the service has a daily limit and its answers are only partly cached in `common/`.

In [1]:
import sys
from pathlib import Path

# The notebook runs from labs/, so the workspace root is the parent folder.
ROOT = Path.cwd().parent

DATA_DIR = ROOT / "data" / "test_gold"
NLTK_DIR = ROOT / "models" / "nltk_data"
COMMON_DIR = ROOT / "common"

# TextServer (FreeLing) is off: the service has a daily limit, and only part of the answers are cached in common/.
# Set it to True to add TextServer to all the comparisons.
RUN_TEXTSERVER = False

import json
from collections import Counter
from getpass import getpass

import pandas as pd
import nltk
import requests
import spacy
from nltk.metrics.distance import jaccard_distance
from scipy.stats import pearsonr

# NLTK packages needed by this notebook.
nltk.data.path.insert(0, str(NLTK_DIR))
nltk.download("punkt_tab", download_dir=NLTK_DIR)

# TextServer client (textserver.py in common/)
sys.path.insert(0, str(COMMON_DIR))
from textserver import TextServer

def textserver_login(service):
    """TextServer with the user and password of credentials.json in common/, or asked for if it is not there."""
    credentials_file = COMMON_DIR / "credentials.json"
    if credentials_file.exists():
        credentials = json.loads(credentials_file.read_text())
    else:
        credentials = {"user": input("TextServer user: "), "password": getpass("TextServer password: ")}
    return TextServer(credentials["user"], credentials["password"], service)

def textserver_cached(service, sentences, cache_file):
    """sentence -> list of tokens given by TextServer, querying only the sentences not in cache_file yet."""
    answers = json.loads(cache_file.read_text()) if cache_file.exists() else {}
    missing = [s for s in sorted(set(sentences)) if s not in answers]
    if missing:
        ts = textserver_login(service)
        for sentence in missing:
            # The server sometimes fails or does not answer, so try up to 3 times.
            for attempt in range(3):
                try:
                    answer = getattr(ts, service)(sentence)
                    break
                except requests.RequestException:
                    if attempt == 2:
                        raise
            # TextServer may split a sentence in two, so join the tokens of all its sentences.
            answers[sentence] = [token for sent in answer for token in sent]
            # Save after every query, so an interrupted run does not lose the answers.
            cache_file.write_text(json.dumps(answers, ensure_ascii=False, indent=1))
    print(f"TextServer {service}: {len(set(sentences))} distinct sentences, queried now: {len(missing)}")
    return answers

# SpaCy model needed by this notebook (only its tokenizer is used here).
if not spacy.util.is_package("en_core_web_sm"):
    spacy.cli.download("en_core_web_sm")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/models/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Data

In [2]:
# Load sentence pairs and gold-standard similarity scores
dt = pd.read_csv(DATA_DIR / 'STS.input.SMTeuroparl.txt', sep='\t', header=None)
dt['gs'] = pd.read_csv(DATA_DIR / 'STS.gs.SMTeuroparl.txt', sep='\t', header=None)
print(f"Sentence pairs: {len(dt)}")

Sentence pairs: 459


## Jaccard similarity of the words

Each sentence is represented as the set of its words, and two sentences are compared with the Jaccard coefficient: the number of shared words divided by the number of different words in the two sentences, |A ∩ B| / |A ∪ B|. It is 1 when the two sets are equal and 0 when they share nothing. It only looks at which words appear, not at how many times or in which order.

`nlp.tokenizer` runs only the tokenizer of the spaCy model, without the POS tagger or the lemmatizer, so the spaCy words are comparable with the NLTK ones.

In [3]:
nlp = spacy.load("en_core_web_sm")
# TextServer tokenizer (FreeLing): one query per distinct sentence, saved in textserver_tokenizer.json.
tokenizers = ["NLTK", "spaCy"]

# The set of words of each sentence, with each tokenizer.
words1 = {"NLTK": [set(nltk.word_tokenize(s)) for s in dt[0]],
          "spaCy": [{t.text for t in nlp.tokenizer(s)} for s in dt[0]]}
words2 = {"NLTK": [set(nltk.word_tokenize(s)) for s in dt[1]],
          "spaCy": [{t.text for t in nlp.tokenizer(s)} for s in dt[1]]}
if RUN_TEXTSERVER:
    ts_tokens = textserver_cached("tokenizer", list(dt[0]) + list(dt[1]), COMMON_DIR / "textserver_tokenizer.json")
    tokenizers.append("TextServer")
    words1["TextServer"] = [set(ts_tokens[s]) for s in dt[0]]
    words2["TextServer"] = [set(ts_tokens[s]) for s in dt[1]]

# NLTK returns the Jaccard distance, so convert it to similarity: similarity = 1 - distance.
for tok in tokenizers:
    dt[tok] = [1 - jaccard_distance(w1, w2) for w1, w2 in zip(words1[tok], words2[tok])]

# The vector of similarities of each tokenizer, next to the gold standard.
dt[['gs'] + tokenizers].head()

,gs,NLTK,spaCy
0,4.50,0.346154,0.346154
1,5.00,0.785714,0.785714
2,4.25,0.391304,0.391304
3,4.50,0.545455,0.600000
4,5.00,1.000000,1.000000


## Pearson correlation

In [4]:
# Compare the Jaccard similarities with the gold-standard scores
for tok in tokenizers:
    print(f"Pearson correlation, {tok} words: {pearsonr(dt['gs'], dt[tok])[0]:.4f}")

Pearson correlation, NLTK words: 0.4507
Pearson correlation, spaCy words: 0.4606


## Distribution of the scores

Compare the range of the gold scores with the range of the similarities, and the mean similarity for each range of gold scores.

In [5]:
# Distribution of the gold standard and of the Jaccard similarities.
print(dt[['gs'] + tokenizers].describe().round(3))
print(f"\nPairs with gold score >= 4: {(dt['gs'] >= 4).mean():.1%}\n")

# Mean Jaccard similarity for each range of gold scores.
gold_range = pd.cut(dt['gs'], [0, 3, 4, 4.5, 5], include_lowest=True)
pd.concat({tok: dt.groupby(gold_range, observed=True)[tok].agg(['count', 'mean', 'min', 'max']) for tok in tokenizers}, axis=1).round(3)

            gs     NLTK    spaCy
count  459.000  459.000  459.000
mean     4.550    0.500    0.505
std      0.516    0.243    0.244
min      1.500    0.053    0.053
25%      4.250    0.319    0.327
50%      4.750    0.435    0.444
75%      5.000    0.600    0.617
max      5.000    1.000    1.000

Pairs with gold score >= 4: 90.6%



NLTK                      spaCy                     
              count   mean    min    max count   mean    min    max
gs                                                                 
(-0.001, 3.0]    13  0.374  0.095  0.538    13  0.393  0.095  0.545
(3.0, 4.0]       61  0.342  0.111  0.750    61  0.339  0.111  0.750
(4.0, 4.5]      123  0.395  0.053  0.900   123  0.396  0.053  0.889
(4.5, 5.0]      262  0.593  0.182  1.000   262  0.601  0.222  1.000

## Examples

Pairs where the word overlap and the gold score disagree, and a pair with differences of case and inflection.

In [6]:
def show_pair(i):
    """Print a sentence pair with its gold score, and the Jaccard similarity and word sets of each tokenizer."""
    print(f"Pair {i}  (gold {dt['gs'][i]})")
    print(f"  S1: {dt[0][i]}")
    print(f"  S2: {dt[1][i]}")
    for tok in tokenizers:
        w1, w2 = words1[tok][i], words2[tok][i]
        print(f"  {tok} words (Jaccard {dt[tok][i]:.3f})")
        print(f"    shared:  {sorted(w1 & w2)}")
        print(f"    only S1: {sorted(w1 - w2)}")
        print(f"    only S2: {sorted(w2 - w1)}")
    print()

# Paraphrases with the maximum gold score but low word overlap.
print("=== Gold 5, lowest Jaccard (NLTK) ===\n")
for i in dt[dt['gs'] == 5].nsmallest(2, 'NLTK').index:
    show_pair(i)

# Pairs with a low gold score but a relatively high word overlap.
print("=== Gold < 3, highest Jaccard (NLTK) ===\n")
for i in dt[dt['gs'] < 3].nlargest(2, 'NLTK').index:
    show_pair(i)

# Pair 12: same meaning, but different case and inflection.
print("=== Case and inflection ===\n")
show_pair(12)

=== Gold 5, lowest Jaccard (NLTK) ===

Pair 382  (gold 5.0)
  S1: That provision could open the door wide to arbitrariness.
  S2: This point of the regulations opens the door to the arbitrary one.
  NLTK words (Jaccard 0.235)
    shared:  ['.', 'door', 'the', 'to']
    only S1: ['That', 'arbitrariness', 'could', 'open', 'provision', 'wide']
    only S2: ['This', 'arbitrary', 'of', 'one', 'opens', 'point', 'regulations']
  spaCy words (Jaccard 0.235)
    shared:  ['.', 'door', 'the', 'to']
    only S1: ['That', 'arbitrariness', 'could', 'open', 'provision', 'wide']
    only S2: ['This', 'arbitrary', 'of', 'one', 'opens', 'point', 'regulations']

Pair 385  (gold 5.0)
  S1: That provision could open the door wide to arbitrariness.
  S2: This point of the regulations opens the door to arbitrary power.
  NLTK words (Jaccard 0.235)
    shared:  ['.', 'door', 'the', 'to']
    only S1: ['That', 'arbitrariness', 'could', 'open', 'provision', 'wide']
    only S2: ['This', 'arbitrary', 'of', 'ope

## NLTK vs. spaCy tokenizer

The two tokenizers split some strings differently. Count the pairs whose similarity changes, list the tokens that only one of them produces, and show the pairs with the largest differences.

In [7]:
diff = dt['spaCy'] - dt['NLTK']
print(f"spaCy higher than NLTK: {(diff > 0).sum()}, lower: {(diff < 0).sum()}, equal: {(diff == 0).sum()}\n")

# Tokens produced by only one of the tokenizers (all sentences).
only_nltk, only_spacy = Counter(), Counter()
for s in list(dt[0]) + list(dt[1]):
    nltk_tokens, spacy_tokens = nltk.word_tokenize(s), [t.text for t in nlp.tokenizer(s)]
    only_nltk.update(set(nltk_tokens) - set(spacy_tokens))
    only_spacy.update(set(spacy_tokens) - set(nltk_tokens))
print("Only NLTK: ", ", ".join(f"{t} {n}" for t, n in only_nltk.most_common(10)))
print("Only spaCy:", ", ".join(f"{t} {n}" for t, n in only_spacy.most_common(10)))
print()

# Largest differences; many pairs share the same first sentence, so each first sentence is shown once.
x = dt.assign(diff=diff)
for title, ascending in [("spaCy higher", False), ("NLTK higher", True)]:
    print(f"=== {title} ===\n")
    for i in x.sort_values('diff', ascending=ascending, kind='stable').drop_duplicates(subset=[0]).index[:2]:
        show_pair(i)

spaCy higher than NLTK: 32, lower: 20, equal: 407

Only NLTK:  Maij-Weggen 33, . 25, p.m 25, A5-0323/2000 22, A5-0241/2000 22, A5-0323 10, A5-0241 10, Report/ratio 2, Thank-You 1, report/ratio 1
Only spaCy: - 68, A5 64, Maij 33, Weggen 33, p.m. 25, 0323/2000 22, 0241/2000 22, 0323 10, 0241 10, / 3

=== spaCy higher ===

Pair 172  (gold 4.75)
  S1: Maij-Weggen report (A5-0323/2000)
  S2: Report Maij-Weggen In particular A5-0323 -- 2000)
  NLTK words (Jaccard 0.182)
    shared:  [')', 'Maij-Weggen']
    only S1: ['(', 'A5-0323/2000', 'report']
    only S2: ['--', '2000', 'A5-0323', 'In', 'Report', 'particular']
  spaCy words (Jaccard 0.357)
    shared:  [')', '-', 'A5', 'Maij', 'Weggen']
    only S1: ['(', '0323/2000', 'report']
    only S2: ['--', '0323', '2000', 'In', 'Report', 'particular']

Pair 68  (gold 4.5)
  S1: Van Orden Report (A5-0241/2000)
  S2: Van Orden report In particular A5-0241 -- 2000)
  NLTK words (Jaccard 0.250)
    shared:  [')', 'Orden', 'Van']
    only S1: ['(', '

## Justification

The Pearson correlation between the Jaccard similarities and the gold standard is **0.4507** with the NLTK words and **0.4606** with the spaCy words: a positive but moderate correlation in both cases. Word overlap captures part of the similarity, but far from all of it. The analysis below uses the NLTK words (the tokenizer of the lab slides); the difference between the two tokenizers is discussed at the end.

**Why it is positive.** Pairs with more words in common tend to have higher gold scores. The pairs with a gold score above 4.5 have a mean Jaccard similarity of 0.593, against 0.342–0.395 for the other ranges, and pair 4, where both sentences have the same set of words, gets both a gold score of 5 and a Jaccard similarity of 1.

**Why it is only moderate.**

- **Paraphrases with different words.** Pair 382 has the maximum gold score (5.0), but a Jaccard similarity of only 0.235: *provision* / *point of the regulations*, *open* / *opens* and *arbitrariness* / *arbitrary* mean the same, but they are different words, and only *door*, *the*, *to* and *.* are shared.
- **Surface variants count as different words.** The comparison is case-sensitive and uses inflected forms. In pair 12 (gold 4.75), *the* / *The* and *adopted* / *adopts* are counted as different words, so the similarity is only 0.556 (with both tokenizers).
- **Function words and punctuation count as much as content words.** In pair 392 (gold 2.5, Jaccard 0.444), the shared words are only *Thank*, *you*, *,* and *.*, while the words that carry the meaning (*Commissioner* vs *Police chief*) are different.
- **The gold standard is concentrated at the top of the scale.** The mean gold score is 4.55 and 90.6% of the pairs have a score of 4 or more (standard deviation 0.516), while the Jaccard similarity spreads from 0.053 to 1.0. Most pairs are near-paraphrases with very similar gold scores but very different word overlaps, which limits the correlation. The relation is not even monotonic at the bottom: the pairs with a gold score of 3 or less have a higher mean Jaccard similarity (0.374) than those between 3 and 4 (0.342).

**NLTK vs. spaCy tokenizer.** The two tokenizers give the same similarity for 407 of the 459 pairs; spaCy is higher for 32 and lower for 20. The differences come from a few kinds of strings that are frequent in this corpus:

- **Hyphens and reference codes.** spaCy splits *Maij-Weggen* into *Maij*, *-*, *Weggen* and *A5-0323/2000* into *A5*, *-*, *0323/2000*, while NLTK keeps them as one token. This helps when a reference is written differently in the two sentences: in pair 172 (gold 4.75) *A5-0323/2000* / *A5-0323 -- 2000* share *A5* with spaCy, and the similarity rises from 0.182 to 0.357. But the shared *-* carries no meaning.
- **Abbreviations at the end of the sentence.** NLTK splits *p.m.* into *p.m* and *.*, spaCy keeps *p.m.* as one token. In pair 9 (gold 5.0) the final full stop is then only shared with NLTK, and the similarity drops from 0.636 to 0.545 with spaCy.

So the tokenizer changes the correlation a little (+0.0099 for spaCy), mostly through names and reference codes, but it does not solve any of the problems above.

So the Jaccard similarity of words is a reasonable baseline, but it only measures surface overlap. Normalizing the words (lowercasing, lemmatization), ignoring stop words and punctuation, or using lexical resources for synonyms could improve the correlation.